<a href="https://colab.research.google.com/github/jccrews256/ST-554-HW-6/blob/main/Homework_6_Part_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 6 Part I

*Class: ST 554*

*Author: Cass Crews*

For Part I of Homework 6, we will continue to query a database that contains information on Major League Baseball.

Before we connect to the database and begin our queries, we need to read in the modules we'll need.

In [1]:
# Reading in required modules
import pandas as pd
import math
import numpy as np
import sqlite3

### Problem 1

We are now ready to connect to the database. We'll utilize the `sqlite3` module to make the connection. After we connect to the dataset, we will query the schema to identify each table in the database.

*Note to reader: The file is too large to be stored in a GitHub repository, so it will need to be brought into your own Colab environment if you are attempting to recreate my results.*

In [2]:
# Make connection to database file; I stored the file "locally" in the colab
# environment
con = sqlite3.connect("lahman_1871-2022.sqlite")

# Constructing SQL query to access tables in schema
get_schema = """
    SELECT *
    FROM sqlite_schema
    WHERE type = "table";
    """

# Obtaining tables from schema
baseball_tables = pd.read_sql(get_schema, con)

# Printing tables
baseball_tables

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


Note that we have a tremendous amount of information on teams, players, and other key team members (e.g., managers), ranging from ballpark characteristics to information on Hall of Fame members.

### Problem 2

Let's query a few of these tables. To start, we will identify pitchers who are in the Hall of Fame and aggregate a few of their pitching statistics to the career level. This will give us an understanding of what it takes to become a Hall of Fame pitcher. To create this data frame, we will join the `HallOfFame` and `Pitching` tables after subsetting to rows in the Hall of Fame table that actually indicate induction rather than simply being up for induction (`inducted = "Y"`). `Pitching` contains season-level statistics, so we will generate career-level statistics by grouping by `playerID` and summing the season-level statistics.

To allow for immediate insights, we'll order the pitchers in the data frame by games pitched (`G`). We'll also pull the pitchers' names from the `People` table so we know who these pitchers are.

In [6]:
# Constructing a SQL query to obtain career-level data for each
# pitcher in the HoF
get_pitcher_careers = """
    SELECT Pitching.playerID, nameLast, nameGiven,
    sum(GS) as GS, sum(G) as G,
    sum(W) as W, sum(L) as L, sum(IPOuts) as IPOuts,
    sum(CG) as CG, sum(SHO) as SHO, sum(SV) as SV
    FROM Pitching
    INNER JOIN HallOfFame on Pitching.playerID = HallOfFame.playerID
    LEFT JOIN People on Pitching.playerID = People.playerID
    WHERE inducted = "Y"
    GROUP BY Pitching.playerID, nameLast, nameGiven
    ORDER BY G DESC;
    """

# Reading career-level pitching data into a data frame
hof_pitcher_careers = pd.read_sql(get_pitcher_careers, con)

# Printing the data frame
hof_pitcher_careers

,playerID,nameLast,nameGiven,GS,G,W,L,IPOuts,CG,SHO,SV
0,riverma01,Rivera,Mariano,10,1115,82,60,3851,0,0,652
1,eckerde01,Eckersley,Dennis Lee,361,1071,197,171,9857,100,20,390
2,wilheho01,Wilhelm,James Hoyt,52,1070,143,122,6763,20,5,227
3,hoffmtr01,Hoffman,Trevor William,0,1035,61,75,3268,0,0,601
4,smithle02,Smith,Lee Arthur,6,1022,71,92,3868,0,0,478
...,...,...,...,...,...,...,...,...,...,...,...
103,hoopeha01,Hooper,Harry Bartholomew,0,1,0,0,6,0,0,0
104,kellyge01,Kelly,George Lange,0,1,1,0,15,0,0,0
105,musiast01,Musial,Stanley Frank,0,1,0,0,0,0,0,0
106,speaktr01,Speaker,Tristram Edgar,0,1,0,0,3,0,0,0


We shouldn't be surprised to see Mariano Rivera at the top of the table. He was a legendary closer, and closers play more frequently than starting pitchers.